## Set up Libraries

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import datetime as dt
from IPython.display import Markdown, display
from lets_plot import *

LetsPlot.setup_html()

## Input Parameters

In [2]:
tickers = ["SPY", "IVV", "VOO", "QQQ", "VTI", "IWM", "XLK", "SOXX"]
rf_ticker = "^IRX"  # CBOE 13-week Treasury yield

# Define by period or by specific start date.
# Set exactly one of the two below; leave the other as None.
period = 365  # In days
start = dt.datetime.now() - dt.timedelta(days=period)  # Start date for data retrieval

print("period:", period)
print("start:", start)

period: 365
start: 2025-09-28 19:23:07.545770


In [3]:
# Constant parameters
trading_days = 252  # Number of trading days in a year
lam = 1.0  # Risk aversion parameter

## Pull data from yfinance

In [4]:
risk_free_tickers = [
    "^TNX",  # CBOE 10-year Treasury yield
    "^FVX",  # CBOE 5-year Treasury yield
    "^IRX",  # 13 WEEK TREASURY BILL (3-month)
]

# ETF to replace the factors in the Fama-French model
factors = [
    "^GSPC",  # Market Factor: S&P 500 index
    "IWM",  # Size Factor: Russell 2000 index
    "IVE", "IVW",  # Value Factor: S&P 500 Value and Growth indices
    "MTUM"  # Momentum Factor: S&P 500 Momentum index
]

In [5]:
# Keep the price levels as well as the returns: ^IRX is a yield, and its level is the risk-free rate
prices = (
    yf.download(
        tickers + factors + risk_free_tickers,
        start=start,
        auto_adjust=True,
        progress=False,
    )["Close"]
    .dropna()
)
returns = prices.pct_change().dropna()

### Portfolio returns

In [6]:
R = returns[tickers]
R

Ticker,SPY,IVV,VOO,QQQ,VTI,IWM,XLK,SOXX
Date,,,,,,,,
2025-09-30,0.003767,0.003809,0.003688,0.002739,0.003271,0.001407,0.005458,0.008143
2025-10-01,0.003408,0.003795,0.003576,0.004797,0.003474,0.002190,0.009792,0.019253
2025-10-02,0.001152,0.000982,0.001106,0.004111,0.001458,0.006310,0.005411,0.019360
2025-10-03,-0.000015,-0.000119,0.000081,-0.004210,0.000546,0.007417,-0.005032,-0.006141
2025-10-06,0.003586,0.003673,0.003413,0.007510,0.003758,0.003986,0.010115,0.030004
...,...,...,...,...,...,...,...,...
2026-09-21,0.015505,0.015609,0.015674,0.028821,0.015103,0.005209,0.028889,0.049281
2026-09-22,-0.000155,-0.000064,0.000000,0.008079,0.000446,0.005708,0.007288,0.024028
2026-09-23,-0.007202,-0.007209,-0.007267,-0.008362,-0.007973,-0.018419,-0.004738,-0.012326


### Factor returns

$$
\text{MKT}_t = R_{m,t} - R_{f,t}, \qquad
\text{SMB}_t = R_{\text{small},t} - R_{\text{big},t}, \qquad
\text{HML}_t = R_{\text{value},t} - R_{\text{growth},t}, \qquad
\text{UMD}_t = R_{\text{mom},t}
$$

$$
R_{i,t} - R_{f,t} = \alpha_i + \beta_{i,\text{MKT}}\,\text{MKT}_t + \beta_{i,\text{SMB}}\,\text{SMB}_t + \beta_{i,\text{HML}}\,\text{HML}_t + \beta_{i,\text{UMD}}\,\text{UMD}_t + \epsilon_{i,t}
$$

In [7]:
# Factor returns
# Rename to match the Fama-French factors
returns["MKT"] = returns["^GSPC"] - returns[rf_ticker]  # Market factor: excess return over risk-free rate
returns["SMB"] = returns["IWM"] - returns["^GSPC"]  # Size factor: small minus big
returns["HML"] = returns["IVE"] - returns["IVW"]  # Value factor: high minus low
returns["UMD"] = returns["MTUM"]  # Momentum factor: up minus down
F = returns[["MKT", "SMB", "HML", "UMD"]]
F

Ticker,MKT,SMB,HML,UMD
Date,,,,
2025-09-30,0.008750,-0.002684,-0.001211,0.003168
2025-10-01,0.003400,-0.001210,-0.001706,-0.002145
2025-10-02,-0.000162,0.005691,-0.001402,-0.000352
2025-10-03,-0.002533,0.007352,0.006713,-0.006020
2025-10-06,0.003128,0.000340,-0.002482,-0.000236
...,...,...,...,...
2026-09-21,0.013922,-0.009718,-0.016394,0.021413
2026-09-22,-0.005784,0.005715,-0.002782,0.005217
2026-09-23,-0.013291,-0.010870,0.004299,-0.002988


# Σ = BΩBᵀ + D equation

# Estimate mu

# Regress returns on the factors

In [8]:
# Estimate annualised expected returns
mu =\
(
    R
    .mean()
    * 252
)
mu.round(4)

Ticker
SPY     0.1715
IVV     0.1724
VOO     0.1720
QQQ     0.2453
VTI     0.1688
IWM     0.1840
XLK     0.3814
SOXX    0.8774
dtype: float64

In [9]:
# Factor loadings B - How much of each factor an asset carries.
B = pd.DataFrame()
for factor in F.columns:
    B[factor] = [R[ticker].cov(F[factor]) / F[factor].var() for ticker in tickers]
B.index = tickers
B

,MKT,SMB,HML,UMD
SPY,0.714964,0.272631,-0.561908,0.373406
IVV,0.715018,0.267107,-0.560414,0.371027
VOO,0.711744,0.270430,-0.559650,0.370696
QQQ,0.996762,0.310966,-1.055738,0.660194
VTI,0.730156,0.365462,-0.547048,0.379504
IWM,0.883160,1.287533,-0.476605,0.489873
XLK,1.162834,0.438659,-1.442709,0.899417
SOXX,1.706657,1.261442,-2.162778,1.635076


In [10]:
# Factor covariance Ω - Coveraince matrix for the factors
Omega = F.cov()
Omega

Ticker,MKT,SMB,HML,UMD
Ticker,,,,
MKT,0.000097,0.000016,-0.000047,0.000097
SMB,0.000016,0.000046,0.000007,0.000034
HML,-0.000047,0.000007,0.000091,-0.000115
UMD,0.000097,0.000034,-0.000115,0.000282


In [11]:
# Regression residuals: Actual Returns R - Predicted Returns (F @ B.T)
residuals = R - F.values @ B.values.T
residuals.round(6)

Ticker,SPY,IVV,VOO,QQQ,VTI,IWM,XLK,SOXX
Date,,,,,,,,
2025-09-30,-0.003621,-0.003585,-0.003667,-0.008518,-0.004002,-0.004995,-0.008136,-0.011204
2025-10-01,0.001148,0.001527,0.001323,0.001399,0.001314,0.000982,0.005836,0.014793
2025-10-02,-0.000941,-0.001077,-0.000972,0.001254,-0.001138,-0.001371,0.001396,0.010000
2025-10-03,0.005812,0.005724,0.005885,0.007091,0.005666,0.006337,0.009788,0.013270
2025-10-06,-0.000049,0.000043,-0.000207,0.001822,0.000082,-0.000281,0.002961,0.019255
...,...,...,...,...,...,...,...,...
2026-09-21,-0.009007,-0.008881,-0.008719,-0.013479,-0.008605,-0.012877,-0.025948,-0.032689
2026-09-22,-0.001089,-0.000950,-0.000920,0.005685,-0.000921,-0.000425,0.002800,0.012143
2026-09-23,0.008795,0.008716,0.008646,0.014778,0.009190,0.010828,0.024375,0.038253


In [12]:
# Idiosyncratic covariance
D = residuals.cov()
D.round(6)

Ticker,SPY,IVV,VOO,QQQ,VTI,IWM,XLK,SOXX
Ticker,,,,,,,,
SPY,0.000095,0.000095,0.000094,0.000145,0.000096,0.000116,0.000185,0.000293
IVV,0.000095,0.000094,0.000094,0.000145,0.000096,0.000116,0.000184,0.000292
VOO,0.000094,0.000094,0.000094,0.000145,0.000096,0.000116,0.000184,0.000292
QQQ,0.000145,0.000145,0.000145,0.000239,0.000148,0.000182,0.000308,0.000503
VTI,0.000096,0.000096,0.000096,0.000148,0.000098,0.000118,0.000189,0.000298
IWM,0.000116,0.000116,0.000116,0.000182,0.000118,0.000146,0.000234,0.000372
XLK,0.000185,0.000184,0.000184,0.000308,0.000189,0.000234,0.000415,0.000666
SOXX,0.000293,0.000292,0.000292,0.000503,0.000298,0.000372,0.000666,0.001190


assemble Σ = BΩBᵀ + D

In [13]:
S = 252 * (
    B.values @ Omega.values @ B.values.T
    + D.values
)

S = pd.DataFrame(
    S,
    index=tickers,
    columns=tickers
)

S.round(6)

,SPY,IVV,VOO,QQQ,VTI,IWM,XLK,SOXX
SPY,0.091580,0.091298,0.091077,0.147774,0.093083,0.115644,0.191675,0.312919
IVV,0.091298,0.091044,0.090797,0.147253,0.092783,0.115204,0.190975,0.311687
VOO,0.091077,0.090797,0.090592,0.146973,0.092587,0.115023,0.190667,0.311190
QQQ,0.147774,0.147253,0.146973,0.243736,0.150064,0.185028,0.317436,0.521178
VTI,0.093083,0.092783,0.092587,0.150064,0.094787,0.118824,0.194743,0.318413
IWM,0.115644,0.115204,0.115023,0.185028,0.118824,0.159122,0.240449,0.400492
XLK,0.191675,0.190975,0.190667,0.317436,0.194743,0.240449,0.418566,0.684228
SOXX,0.312919,0.311687,0.311190,0.521178,0.318413,0.400492,0.684228,1.155218


# Run naive MVO

In [14]:
# Naive MVO
Sigma_np = S.values
n = len(tickers)

# KKT system
KKT = np.block([
    [lam * Sigma_np, np.ones((n, 1))],
    [np.ones((1, n)), np.zeros((1, 1))]
])

rhs = np.concatenate([
    mu,
    [1.0]
])

solution = np.linalg.solve(KKT, rhs)

w_star = solution[:n]

print("Naive MVO weights:")
print(pd.Series(w_star, index=tickers).round(4))

print("\nSum of weights:", round(w_star.sum(), 6))
print("Gross exposure:", round(np.abs(w_star).sum(), 6))

Naive MVO weights:
SPY     -98.8101
IVV      17.4765
VOO     165.7103
QQQ     -58.7009
VTI     -32.9807
IWM     -11.1563
XLK       4.6143
SOXX     14.8469
dtype: float64

Sum of weights: 1.0
Gross exposure: 404.296092


## Why the naive MVO is one linear solve: the KKT conditions

The naive MVO problem is

$$
\min_{w}\; \tfrac{\lambda}{2}\, w^\top \Sigma\, w \;-\; \mu^\top w
\qquad \text{subject to} \qquad \mathbf{1}^\top w = 1
$$

[Tibshirani's KKT lecture](https://www.stat.cmu.edu/~ryantibs/convexopt/lectures/kkt.pdf) (CMU 10-725) gives four conditions for $\min_x f(x)$ subject to $h_i(x) \le 0$ and $\ell_j(x) = 0$. Here is what each one becomes for this problem:

| KKT condition | General form | Naive MVO |
|---|---|---|
| Stationarity | $0 \in \partial f(x) + \sum_i u_i\, \partial h_i(x) + \sum_j v_j\, \partial \ell_j(x)$ | $\lambda \Sigma w - \mu + \nu \mathbf{1} = 0$ |
| Complementary slackness | $u_i\, h_i(x) = 0$ | none: there are no inequality constraints (no long-only rule, no leverage cap) |
| Primal feasibility | $h_i(x) \le 0,\; \ell_j(x) = 0$ | $\mathbf{1}^\top w = 1$ |
| Dual feasibility | $u_i \ge 0$ | none: no inequality multipliers |

**Why solving the KKT conditions gives the optimum.** The lecture shows that KKT is *always sufficient*, and *necessary* when strong duality holds. Here the objective is convex ($\Sigma \succeq 0$, $\lambda > 0$) and the only constraint is affine. That is enough for the refined Slater condition, so strong duality holds and

$$w^\star \text{ is optimal} \iff (w^\star, \nu^\star) \text{ satisfies the KKT conditions.}$$

$\Sigma$ is positive definite, so the solution is also unique.

**From conditions to code.** This is the lecture's *"quadratic with equality constraints"* example, with $Q = \lambda\Sigma$, $c = -\mu$ and $A = \mathbf{1}^\top$. Stationarity and primal feasibility stack into one linear system, which is exactly the `KKT` matrix and `rhs` vector in the cell above:

$$
\begin{bmatrix} \lambda \Sigma & \mathbf{1} \\ \mathbf{1}^\top & 0 \end{bmatrix}
\begin{bmatrix} w \\ \nu \end{bmatrix}
=
\begin{bmatrix} \mu \\ 1 \end{bmatrix}
\quad\Longrightarrow\quad
w^\star = \tfrac{1}{\lambda}\,\Sigma^{-1}\!\left(\mu - \nu^\star \mathbf{1}\right),
\qquad
\nu^\star = \frac{\mathbf{1}^\top \Sigma^{-1} \mu - \lambda}{\mathbf{1}^\top \Sigma^{-1} \mathbf{1}}
$$

The multiplier $\nu^\star$ is the shadow price of the budget constraint: how much the objective would improve if one more unit of capital were allowed.

**Why the weights explode.** $w^\star$ depends on $\mu$ through $\Sigma^{-1}$. SPY, IVV, VOO and VTI are almost perfectly correlated (see the correlation heatmap below), so $\Sigma$ is close to singular and $\Sigma^{-1}$ turns tiny differences in $\mu$ into huge offsetting long and short positions. The KKT system has nothing to stop this. With no inequality constraints, no complementary-slackness condition can bind and cap the leverage. The ridge penalty below fixes the conditioning directly: it replaces $\lambda\Sigma$ with $\lambda\Sigma + \eta I$.

In [15]:
# Check that the KKT conditions hold at the solution
nu_star = solution[n]

stationarity = lam * Sigma_np @ w_star - np.asarray(mu) + nu_star * np.ones(n)

print("ν* (shadow price of the budget):", round(nu_star, 6))
print("Stationarity  ‖λΣw* − μ + ν*1‖:", f"{np.linalg.norm(stationarity):.2e}")
print("Primal feasibility  1ᵀw* − 1:", f"{w_star.sum() - 1:.2e}")
print("Condition number of Σ:", f"{np.linalg.cond(Sigma_np):,.0f}")

ν* (shadow price of the budget): 0.036795
Stationarity  ‖λΣw* − μ + ν*1‖: 5.61e-15
Primal feasibility  1ᵀw* − 1: -7.11e-15
Condition number of Σ: 390,750


# Nudge μ by 5 basis points

In [16]:
# Small expected-return error: +5 basis points to SPY
mu = R.mean().values * 252
mu_nudged = mu.copy()
mu_nudged[0] += 0.0005

# Solve fully-invested MVO with nudged expected returns
rhs_nudged = np.concatenate([
    mu_nudged,
    [1.0]
])

solution_nudged = np.linalg.solve(
    KKT,
    rhs_nudged
)

w_nudged = solution_nudged[:n]

# Weight movement
weight_change = w_nudged - w_star

print("Original weights:")
print(pd.Series(w_star, index=tickers).round(4))

print("\nNudged weights:")
print(pd.Series(w_nudged, index=tickers).round(4))

print("\nWeight change:")
print(pd.Series(weight_change, index=tickers).round(4))

print("\nTotal weight displacement:",
      round(np.linalg.norm(weight_change), 4))

print("\nOriginal sum:", round(w_star.sum(), 6))
print("Nudged sum:", round(w_nudged.sum(), 6))

Original weights:
SPY     -98.8101
IVV      17.4765
VOO     165.7103
QQQ     -58.7009
VTI     -32.9807
IWM     -11.1563
XLK       4.6143
SOXX     14.8469
dtype: float64

Nudged weights:
SPY     -46.2396
IVV      -0.6505
VOO     133.6241
QQQ     -59.3396
VTI     -34.9829
IWM     -11.1093
XLK       4.9054
SOXX     14.7924
dtype: float64

Weight change:
SPY     52.5705
IVV    -18.1270
VOO    -32.0862
QQQ     -0.6387
VTI     -2.0022
IWM      0.0470
XLK      0.2911
SOXX    -0.0545
dtype: float64

Total weight displacement: 64.2361

Original sum: 1.0
Nudged sum: 1.0


# Add ridge penalty

In [17]:
etas = [0, 0.01, 0.05, 0.10, 0.50, 1.00]

I = np.eye(n)
ones = np.ones((n, 1))

results = []

In [18]:
for eta in etas:

    # KKT system for fully-invested ridge MVO
    KKT_ridge = np.block([
        [lam * S.values + eta * I, ones],
        [ones.T, np.zeros((1, 1))]
    ])

    # Original expected returns
    rhs = np.concatenate([
        mu,
        [1.0]
    ])

    solution = np.linalg.solve(KKT_ridge, rhs)
    w_before = solution[:n]

    # Nudge SPY by +5 bps
    mu_nudged = mu.copy()
    mu_nudged[0] += 0.0005

    rhs_nudged = np.concatenate([
        mu_nudged,
        [1.0]
    ])

    solution_nudged = np.linalg.solve(
        KKT_ridge,
        rhs_nudged
    )

    w_after = solution_nudged[:n]

    # Weight swing
    swing = np.linalg.norm(w_after - w_before)

    # Turnover
    turnover = 0.5 * np.abs(w_after - w_before).sum()

    results.append([
        eta,
        swing,
        turnover
    ])

ridge_results = pd.DataFrame(
    results,
    columns=[
        "eta",
        "weight_swing",
        "turnover"
    ]
)

print(ridge_results.round(4))

# Save ridge portfolio when eta = 1
if eta == 1.00:
    w_ridge = w_before.copy()

    eta  weight_swing  turnover
0  0.00       64.2361   52.9086
1  0.01        0.0442    0.0425
2  0.05        0.0090    0.0087
3  0.10        0.0045    0.0043
4  0.50        0.0009    0.0008
5  1.00        0.0005    0.0004


# Go factor neutral - Test the alpha

In [19]:
# Annualised factor mean
factor_mean = F["MKT"].mean() * 252

# Annualised alpha
alpha = mu - B["MKT"].values * factor_mean

print("Annualised alpha:")
print(pd.Series(alpha, index=tickers).round(4))

Annualised alpha:
SPY     0.0965
IVV     0.0974
VOO     0.0973
QQQ     0.1407
VTI     0.0922
IWM     0.0914
XLK     0.2594
SOXX    0.6984
dtype: float64


# Solve the factor-neutral portfolio

In [20]:
D_np = D.values
B_np = B.values
alpha_np = np.asarray(alpha)
n = len(tickers)
m = B_np.shape[1]  # number of factors to neutralize

# Constraints:
# 1. Fully invested: 1'w = 1
# 2. Factor neutral: B'w = 0 (one row per factor)

C = np.column_stack([
    np.ones(n),
    B_np
])

d = np.concatenate([[1.0], np.zeros(m)])

# KKT system
KKT = np.block([
    [lam * D_np, C],
    [C.T, np.zeros((m + 1, m + 1))]
])

rhs = np.concatenate([
    alpha_np,
    d
])

solution = np.linalg.solve(KKT, rhs)

w_fn = solution[:n]

print("Factor-neutral portfolio:")
print(pd.Series(w_fn, index=tickers).round(4))

print("\nSum of weights:")
print(round(w_fn.sum(), 6))

print("\nFactor exposure Bᵀw:")
print((B_np.T @ w_fn).round(8))

from scipy.optimize import minimize

n = len(tickers)

def objective(w):
    return -(alpha @ w - (lam / 2) * w @ D_np @ w)

constraints = [
    {
        "type": "eq",
        "fun": lambda w: w.sum() - 1
    },
    {
        "type": "eq",
        "fun": lambda w: B_np.T @ w
    },
    {
        "type": "ineq",
        "fun": lambda w: 2.0 - np.abs(w).sum()
    }
]

w0 = np.ones(n) / n

result = minimize(
    objective,
    w0,
    method="SLSQP",
    constraints=constraints,
    options={"maxiter": 2000, "ftol": 1e-10}
)

w_fn = result.x

print("Factor-neutral constrained portfolio:")
print(pd.Series(w_fn, index=tickers).round(4))

print("\nSum of weights:", round(w_fn.sum(), 6))
print("Factor exposure:", (B_np.T @ w_fn).round(8))
print("Gross exposure:", round(np.abs(w_fn).sum(), 6))

Factor-neutral portfolio:
SPY    -59810.6435
IVV     37003.6726
VOO     19432.5606
QQQ     -1437.4680
VTI      4500.3004
IWM      -436.9633
XLK       502.4964
SOXX      247.0447
dtype: float64

Sum of weights:
1.0

Factor exposure Bᵀw:
[ 0.  0. -0.  0.]
Factor-neutral constrained portfolio:
SPY     0.0001
IVV     0.0001
VOO     6.3770
QQQ    -9.1754
VTI     0.0000
IWM    -0.6486
XLK     4.4309
SOXX    0.0159
dtype: float64

Sum of weights: 1.0
Factor exposure: [ 4.0e-08 -1.0e-08  1.5e-07 -3.0e-08]
Gross exposure: 20.648071


# Tracking error comparison

In [21]:
# Hypothetical equal-weighted benchmark
b = np.ones(n) / n

In [22]:
# Active weights
a_naive = w_star - b
a_ridge = w_ridge - b
a_fn = w_fn - b

# Tracking error squared
TE2_naive = a_naive @ S.values @ a_naive
TE2_ridge = a_ridge @ S.values @ a_ridge
TE2_fn = a_fn @ S.values @ a_fn

# Annualised tracking error
TE_naive = np.sqrt(TE2_naive)
TE_ridge = np.sqrt(TE2_ridge)
TE_fn = np.sqrt(TE2_fn)

# Comparison
TE_results = pd.DataFrame({
    "Portfolio": [
        "Naive MVO",
        "Ridge MVO (eta=1)",
        "Factor Neutral"
    ],
    "TE²": [
        TE2_naive,
        TE2_ridge,
        TE2_fn
    ],
    "Tracking Error": [
        TE_naive,
        TE_ridge,
        TE_fn
    ]
})

print(TE_results.round(6))

           Portfolio       TE²  Tracking Error
0          Naive MVO  6.994726        2.644754
1  Ridge MVO (eta=1)  0.021029        0.145013
2     Factor Neutral  0.338936        0.582183


# Backtest: holding each portfolio without rebalancing

**There is no rebalancing in this notebook.** Each strategy is solved once on the full sample and never re-solved or reset. The backtest below therefore buys every sleeve at its target weight on the first close and holds it to the end. Because nothing resets the weights, each sleeve's share of the portfolio drifts with its own return. The weights chart shows that drift.

| Strategy | Weights used |
|---|---|
| Naive MVO | `w_star`: fully invested, unconstrained KKT solve on $\Sigma = B\Omega B^\top + D$ |
| Ridge MVO | `w_ridge`: KKT solve on $\lambda\Sigma + \eta I$ with $\eta = 1$ |
| Factor Neutral | `w_fn`: SLSQP on alpha and $D$, subject to $B^\top w = 0$ |

> **Caveat: these results are in-sample.** $\mu$, $B$, $\Sigma$ and alpha are all estimated on the same window the backtest runs over, so the weights "know" the returns they then earn. Treat the returns and alphas below as an upper bound, not a forecast.

In [23]:
# The notebook solves each portfolio once, on the full sample, and never re-solves or resets it:
# there is no rebalancing. So the backtest is buy-and-hold -- every sleeve is bought at its target
# weight on the first close and then left to compound at its own rate.
strategy_weights = pd.DataFrame(
    {
        "Naive MVO": w_star,
        "Ridge MVO": w_ridge,
        "Factor Neutral": w_fn,
    },
    index=tickers,
)
STRATEGIES = list(strategy_weights.columns)

# Growth of $1 in each ticker, starting from the close before the first return (value 1.0)
growth = pd.concat([
    pd.DataFrame(1.0, index=[prices.index[0]], columns=tickers),
    (1 + R).cumprod(),
]).rename_axis("Date")

# Dollar value held in each ticker, per strategy. The weights sum to 1, so every strategy starts at $1
holdings = {s: growth * strategy_weights[s] for s in STRATEGIES}
portfolio_values = pd.DataFrame({s: holdings[s].sum(axis=1) for s in STRATEGIES})

# Drifted weights: each sleeve's share of the strategy's value on that day.
# Winners grow their share and losers shrink theirs, because nothing resets them.
drifted_weights = {s: holdings[s].div(portfolio_values[s], axis=0) for s in STRATEGIES}

# Daily strategy returns, on the same dates as R
portfolio_returns = portfolio_values.pct_change().dropna()

# Daily risk-free rate: ^IRX is an annualised yield in percent, so turn it into a daily rate.
# The yield quoted at yesterday's close is what is earned over today.
rf_daily = (prices[rf_ticker].shift(1) / 100 / trading_days).loc[R.index]

portfolio_values.tail()

,Naive MVO,Ridge MVO,Factor Neutral
Date,,,
2026-09-21,11.821154,1.513880,1.550381
2026-09-22,11.945888,1.533050,1.499651
2026-09-23,11.965960,1.517547,1.525630
2026-09-24,11.914739,1.517071,1.501600
2026-09-25,12.336571,1.529623,1.538444


## The French-Fama benchmark

Each strategy is compared with a benchmark built from its own factor exposures. First, regress the strategy's daily excess return on the four factors:

$$
R_{p,t} - R_{f,t} = \alpha_p + \beta_{p,\text{MKT}}\,\text{MKT}_t + \beta_{p,\text{SMB}}\,\text{SMB}_t + \beta_{p,\text{HML}}\,\text{HML}_t + \beta_{p,\text{UMD}}\,\text{UMD}_t + \varepsilon_{p,t}
$$

The benchmark is the return those betas alone would have earned, with no alpha. The strategy's active return is what is left: alpha plus noise. Tracking error is the volatility of that active return.

$$
R^{\text{FF}}_{p,t} = R_{f,t} + \hat\beta_p^\top F_t,
\qquad
R_{p,t} - R^{\text{FF}}_{p,t} = \hat\alpha_p + \hat\varepsilon_{p,t},
\qquad
\text{TE}_p = \sqrt{252}\;\operatorname{sd}\!\left(R_{p,t} - R^{\text{FF}}_{p,t}\right)
$$

In [24]:
# Regress each strategy's daily excess return on the four factors:
#   R_p,t - R_f,t = alpha + b_MKT MKT_t + b_SMB SMB_t + b_HML HML_t + b_UMD UMD_t + e_t
excess_returns = portfolio_returns.sub(rf_daily, axis=0)

X = np.column_stack([np.ones(len(F)), F.values])  # intercept column + factors
XtX_inv = np.linalg.inv(X.T @ X)
dof = len(F) - X.shape[1]

coefficients = {}
alpha_t_stats = {}
r_squared = {}
for s in STRATEGIES:
    y = excess_returns[s].values
    c = XtX_inv @ X.T @ y
    e = y - X @ c
    coefficients[s] = c
    alpha_t_stats[s] = c[0] / np.sqrt((e @ e / dof) * XtX_inv[0, 0])  # OLS standard error
    r_squared[s] = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))

coefficients = pd.DataFrame(coefficients, index=["alpha"] + list(F.columns))

# French-Fama benchmark: what each strategy's factor exposures alone would have earned,
# i.e. R_f plus the fitted factor return, with no alpha.
betas = coefficients.loc[F.columns]
benchmark_returns = pd.DataFrame(F.values @ betas.values, index=F.index, columns=STRATEGIES).add(rf_daily, axis=0)

benchmark_values = pd.concat([
    pd.DataFrame(1.0, index=[prices.index[0]], columns=STRATEGIES),
    (1 + benchmark_returns).cumprod(),
]).rename_axis("Date")

# Active return = strategy - benchmark = alpha + e_t
active_returns = portfolio_returns - benchmark_returns

coefficients.round(4)

,Naive MVO,Ridge MVO,Factor Neutral
alpha,0.0096,0.0007,0.0017
MKT,-0.9535,0.2578,-0.0684
SMB,-1.9782,0.1072,-0.0459
HML,1.5927,-0.1999,-0.0641
UMD,3.1328,0.8279,0.1623


## Charts

In [25]:
# Correlation of daily returns across the eight tickers and the four factors. Correlation is
# unitless and bounded to [-1, 1], so the colour scale is fixed to that range: -1 always reads
# as fully blue, not just "the most negative cell in this matrix".
# The near-1 block of SPY / IVV / VOO / VTI is why the naive MVO blows up: Σ is close to singular,
# so Σ⁻¹ turns tiny differences in μ into huge offsetting long/short positions.
correlation_assets = tickers + list(F.columns)

correlation = pd.concat([R, F], axis=1)[correlation_assets].corr()

correlation_long = (
    correlation
    .rename_axis(index="Row", columns="Column")
    .stack()
    .rename("Correlation")
    .reset_index()
    # The ink flips to white only where the tile is dark enough to swallow black text
    .assign(
        Label=lambda frame: frame["Correlation"].map("{:.2f}".format),
        Ink=lambda frame: np.where(frame["Correlation"].abs() > 0.5, "white", "#0b0b0b"),
    )
)

correlation_plot = (
    ggplot(correlation_long, aes(x="Column", y="Row"))
    + geom_tile(aes(fill="Correlation"), width=0.96, height=0.96)
    + geom_text(aes(label="Label", color="Ink"), size=7)
    + scale_fill_gradient2(
        low="#2a78d6",   # -1: moves opposite
        mid="#f0efec",   #  0: no linear relationship
        high="#e34948",  # +1: moves together
        midpoint=0,
        limits=[-1, 1],
        name="Correlation",
    )
    + scale_color_identity()  # "Ink" holds the colour itself, not a category
    + scale_x_discrete(limits=correlation_assets)
    + scale_y_discrete(limits=correlation_assets[::-1])  # diagonal top-left -> bottom-right
    + coord_fixed()
    + labs(
        title="Correlation of daily returns: tickers and factors",
        subtitle=f"{R.index[0]:%d %b %Y} to {R.index[-1]:%d %b %Y}  |  the diagonal is each series against itself",
        x="",
        y="",
    )
    + ggsize(800, 700)
)

correlation_plot

In [26]:
# One colour per strategy, used by every chart below
STRATEGY_COLORS = {
    "Naive MVO": "#2a78d6",
    "Ridge MVO": "#eb6834",
    "Factor Neutral": "#1baf7a",
}

equity_long = pd.concat([
    portfolio_values.reset_index()
    .melt(id_vars="Date", var_name="Strategy", value_name="Value")
    .assign(Series="Strategy"),
    benchmark_values.reset_index()
    .melt(id_vars="Date", var_name="Strategy", value_name="Value")
    .assign(Series="French-Fama benchmark"),
])

# Each panel has its own y scale: the naive MVO is ~400x gross levered, so on a shared axis the
# other two strategies would be flat lines along the bottom.
equity_plot = (
    ggplot(equity_long, aes(x="Date", y="Value"))
    + geom_hline(yintercept=1, color="#c3c2b7", size=0.4)
    + geom_line(
        aes(color="Strategy", linetype="Series"),
        data=equity_long.query("Series == 'Strategy'"),
        size=0.9,
    )
    + geom_line(
        aes(linetype="Series"),
        data=equity_long.query("Series == 'French-Fama benchmark'"),
        color="#52514e",
        size=0.7,
    )
    + scale_color_manual(values=STRATEGY_COLORS, name="Strategy")
    + scale_linetype_manual(
        values={"Strategy": "solid", "French-Fama benchmark": "dashed"},
        name="",
    )
    + facet_wrap("Strategy", ncol=1, scales="free_y", order=0)
    + scale_y_continuous(format="$,.2f")
    + labs(
        title="Growth of $1, held without rebalancing, against its French-Fama benchmark",
        subtitle="Benchmark = R_f + fitted factor return (each strategy's own betas, no alpha). "
        "The gap between the lines is alpha plus noise.",
        x="",
        y="Portfolio value",
    )
    + ggsize(1000, 750)
    + theme(legend_position="top")
)

equity_plot

In [27]:
# There is no rebalancing in this notebook: each strategy's weights are solved once and then
# held. So the weights are not fixed over the horizon -- each sleeve drifts with its own return.
# Weights can be negative (short) and far above 100% (leveraged), so they are drawn as lines,
# not a stacked area, with one panel per strategy on its own scale.
TICKER_COLORS = dict(zip(
    tickers,
    ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"],
))

weights_long = pd.concat([
    drifted_weights[s].reset_index()
    .melt(id_vars="Date", var_name="Ticker", value_name="Weight")
    .assign(Strategy=s)
    for s in STRATEGIES
])

weights_plot = (
    ggplot(weights_long, aes(x="Date", y="Weight"))
    + geom_hline(yintercept=0, color="#c3c2b7", size=0.4)
    + geom_line(aes(color="Ticker"), size=0.8)
    + scale_color_manual(values=TICKER_COLORS, name="Ticker")
    + facet_wrap("Strategy", ncol=1, scales="free_y", order=0)
    + scale_y_continuous(format=".0%")
    + labs(
        title="Weights drift over the horizon (no rebalancing)",
        subtitle="Solved once on the first day, then held: each sleeve's share moves with its own return",
        x="",
        y="Weight (share of portfolio value)",
    )
    + ggsize(1000, 750)
    + theme(legend_position="top")
)

weights_plot

# Summary: risk, return, alpha, beta and tracking error

All figures come from the held (never rebalanced) backtest above. Returns and volatility are annualised with 252 trading days, and alpha and betas come from the four-factor regression.

| Metric | Definition |
|---|---|
| Gross exposure | $\sum_i \lvert w_i \rvert$ at inception. 1.0 means no leverage or shorting. |
| Annualised return / volatility | $252 \cdot \bar r_p$ and $\sqrt{252}\,\operatorname{sd}(r_p)$ |
| Sharpe ratio | annualised mean excess return divided by annualised volatility |
| Alpha | annualised regression intercept $252\,\hat\alpha_p$, with its OLS t-stat |
| Betas, R² | four-factor loadings, and the share of variance the factors explain |
| Tracking error vs French-Fama benchmark | $\sqrt{252}\,\operatorname{sd}(R_p - R^{\text{FF}}_p)$ (realised) |
| Information ratio | annualised active return divided by tracking error |
| Ex-ante tracking error vs equal weight | $\sqrt{(w - b)^\top \Sigma\, (w - b)}$ with $b = \mathbf{1}/n$, from the tracking error section above |

In [28]:
def max_drawdown(values: pd.Series) -> float:
    """Largest peak-to-trough fall of a value series, as a negative fraction."""
    return (values / values.cummax() - 1).min()


ann_vol = portfolio_returns.std() * np.sqrt(trading_days)
tracking_error = active_returns.std() * np.sqrt(trading_days)

summary = pd.DataFrame({
    "Gross exposure Σ|w|": strategy_weights.abs().sum(),
    "Total return": portfolio_values.iloc[-1] - 1,
    "Annualised return": portfolio_returns.mean() * trading_days,
    "Annualised volatility": ann_vol,
    "Sharpe ratio": excess_returns.mean() * trading_days / ann_vol,
    "Max drawdown": portfolio_values.apply(max_drawdown),
    "Alpha (annualised)": coefficients.loc["alpha"] * trading_days,
    "Alpha t-stat": pd.Series(alpha_t_stats),
    "Beta MKT": coefficients.loc["MKT"],
    "Beta SMB": coefficients.loc["SMB"],
    "Beta HML": coefficients.loc["HML"],
    "Beta UMD": coefficients.loc["UMD"],
    "R²": pd.Series(r_squared),
    "Tracking error vs French-Fama benchmark": tracking_error,
    "Information ratio": active_returns.mean() * trading_days / tracking_error,
    "Ex-ante tracking error vs equal weight": pd.Series(
        TE_results["Tracking Error"].values, index=STRATEGIES
    ),
}).T

# Percentages for returns and risk; plain numbers for ratios, betas and exposure
percent_rows = [
    "Total return", "Annualised return", "Annualised volatility", "Max drawdown",
    "Alpha (annualised)", "Tracking error vs French-Fama benchmark",
    "Ex-ante tracking error vs equal weight",
]
summary_display = summary.apply(
    lambda row: row.map("{:.2%}".format if row.name in percent_rows else "{:.3f}".format),
    axis=1,
)

summary_display

,Naive MVO,Ridge MVO,Factor Neutral
Gross exposure Σ|w|,404.296,1.000,20.648
Total return,1133.66%,52.96%,53.84%
Annualised return,304.76%,47.04%,49.23%
Annualised volatility,100.33%,28.28%,33.72%
Sharpe ratio,3.001,1.534,1.351
Max drawdown,-32.66%,-17.30%,-17.96%
Alpha (annualised),241.60%,17.38%,41.73%
Alpha t-stat,2.868,2.247,1.228
Beta MKT,-0.953,0.258,-0.068
Beta SMB,-1.978,0.107,-0.046


## The French-Fama benchmark in full

The factors are built from ETFs as proxies for the Fama-French / Carhart factors (see *Factor returns* above):

$$
\text{MKT}_t = R_{\text{S\&P 500},t} - R_{f,t}, \qquad
\text{SMB}_t = R_{\text{IWM},t} - R_{\text{S\&P 500},t}, \qquad
\text{HML}_t = R_{\text{IVE},t} - R_{\text{IVW},t}, \qquad
\text{UMD}_t = R_{\text{MTUM},t}
$$

Each strategy's benchmark return on day $t$ is

$$
R^{\text{FF}}_{p,t} \;=\; R_{f,t} \;+\; \beta_{p,\text{MKT}}\,(R_{\text{S\&P 500},t} - R_{f,t}) \;+\; \beta_{p,\text{SMB}}\,(R_{\text{IWM},t} - R_{\text{S\&P 500},t}) \;+\; \beta_{p,\text{HML}}\,(R_{\text{IVE},t} - R_{\text{IVW},t}) \;+\; \beta_{p,\text{UMD}}\,R_{\text{MTUM},t}
$$

The strategy itself earns that benchmark plus alpha plus noise:

$$
R_{p,t} = R^{\text{FF}}_{p,t} + \alpha_p + \varepsilon_{p,t}
$$

The cell below fills in the fitted betas for each strategy.

In [29]:
# Each strategy's own benchmark, written out with its fitted betas
def signed(value: float) -> str:
    return f"- {abs(value):.4f}" if value < 0 else f"+ {value:.4f}"


equations = []
for s in STRATEGIES:
    b = coefficients[s]
    equations.append(
        rf"**{s}**" "\n\n"
        rf"$$R^{{\text{{FF}}}}_{{t}} = R_{{f,t}} "
        rf"{signed(b['MKT'])}\,\text{{MKT}}_t "
        rf"{signed(b['SMB'])}\,\text{{SMB}}_t "
        rf"{signed(b['HML'])}\,\text{{HML}}_t "
        rf"{signed(b['UMD'])}\,\text{{UMD}}_t$$"
        "\n\n"
        rf"$\alpha$ = {b['alpha'] * trading_days:.2%} p.a. (t = {alpha_t_stats[s]:.2f}), "
        rf"tracking error = {tracking_error[s]:.2%} p.a."
    )

display(Markdown("\n\n".join(equations)))

**Naive MVO**

$$R^{\text{FF}}_{t} = R_{f,t} - 0.9535\,\text{MKT}_t - 1.9782\,\text{SMB}_t + 1.5927\,\text{HML}_t + 3.1328\,\text{UMD}_t$$

$\alpha$ = 241.60% p.a. (t = 2.87), tracking error = 82.83% p.a.

**Ridge MVO**

$$R^{\text{FF}}_{t} = R_{f,t} + 0.2578\,\text{MKT}_t + 0.1072\,\text{SMB}_t - 0.1999\,\text{HML}_t + 0.8279\,\text{UMD}_t$$

$\alpha$ = 17.38% p.a. (t = 2.25), tracking error = 7.60% p.a.

**Factor Neutral**

$$R^{\text{FF}}_{t} = R_{f,t} - 0.0684\,\text{MKT}_t - 0.0459\,\text{SMB}_t - 0.0641\,\text{HML}_t + 0.1623\,\text{UMD}_t$$

$\alpha$ = 41.73% p.a. (t = 1.23), tracking error = 33.42% p.a.

### Reading the summary

- **Naive MVO** has the highest return and the worst risk. At several hundred times gross leverage, its volatility and tracking error dwarf the other two. Its "alpha" is mostly the in-sample fit of an almost singular $\Sigma^{-1}$ to this window's $\mu$, and the 5 bp nudge test above shows how unstable those weights are.
- **Ridge MVO** is unlevered and close to its benchmark. The factors explain most of its variance (high R²), so its tracking error is the lowest of the three. What it earns, it earns mostly through factor exposure.
- **Factor Neutral** is built to have $B^\top w = 0$, so its realised betas are small and R² is low. The factors explain almost none of its return, which is the point. Its tracking error is therefore close to its total volatility, and its alpha is essentially its whole excess return.
  - The realised betas are not exactly zero, for two reasons. The weights drift away from neutral because the portfolio is never rebalanced. And $B$ was estimated one factor at a time ($\operatorname{cov}/\operatorname{var}$) rather than by a joint regression.